__DISCLAIMER 1: This is the first file you should look at, after this please move onto machineLearning.ipynb__

__DISCLAIMER 2: This is the main jupyter notebook file for data cleaning__ which explains all  my thoughts and reasoning as to why I did what I did throughout the entire workflow, as I have worked through the webscraped csv files alphabetically, this one specifically covers the "2bundesliga" folder (representing all data from the "2. Bundesliga" in Germany).

Every other jupyter notebook file under the naming scheme of __"dataCleaning[LEAGUENAME].ipynb"__ follow this exact workflow, without the added commentary __unless__ specified in __this file__, as there may  be differences in cutoffs for statistics, or data erasures or any other league-specific need. (this doesn't apply to naming schemes nor calling for files, as those are assumed to differ naturally)

__DISCLAIMER 3: ENSURE FOLDER STRUCTURE MATCHES folderStructure.PNG__
<br>

After gathering all the data through the use of the webscraper, I manually edited all files to separate player names and teams (this was done slightly manually through adding commas in VScode and then copy and pasting through Excel).

The next step revolves around data exploration of said files, with the goal of merging them into one mega file.

The order of work looks roughly like this:
1. Merge Databases by position and league (i.e. DEFENDER databases in BUNDESLIGA)
2. Add a "position" column to the newly merged dataset
3. After every position was merged together and cleaned up in one league:
    - Merge them all together
    - Add a "league" column
4. Repeat for every league then merge into one final database.

<br>

Before starting, every database was manually renamed to have their respective league in the file name (in the format: LEAGUEpositionSTATAREA.csv), this was done to avoid potential ambiguity in the code and to not have any mistakes come up from the naming scheme.
- (i.e. "defenderPASSING.CSV" located within the premierleague folder -> "PLdefenderPASSING.csv")

<br>

Due to how the website containing the sourced data works, the table was filtered by positions and these general statistical areas: Attacking, Default, Defending, Goalkeeping, Passing.
I made the webscraper scan each filtered table and put the found data into a separate csv, hence resulting in tens of .csv files, with some being useless or with very little use cases, such as forwardGOALKEEPING.csv having the statistics of every forward player in goals (and as expected, 99.99% of them have 0/ n/a in every field)


__Due to the nature of the data-source for the webscraper, some leagues are lacking in full data__ - this is the biggest drawback with public data sources being limited in what they can provide now.

# 1.1 Data Exploration

This section will cover the dataset's features

In [1]:
#importing basics
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from functools import reduce

__This section until 1.1.1 varies in every datacleaning file__

In [2]:
#Dataset initialisation - We will only cover the 2bundesliga datasets initially here (as they are alphabetically first)
#every dataframe for every position will be layed out like:
#                   df<FIRST LETTER OF POSITION CAPITALISED><first three letters of filter lowercase>
#with the exception of default and defending, default = dft, defending = def

dfDdef = pd.read_csv("data/finalSCRAPEDdata/2bundesliga/2BULIdefenderDEFENDING.csv")

#dataset size - will differ for every single dataset, rows should stay consistent inbetween league+position matchings
print(f"df has {dfDdef.shape[0]} rows, and {dfDdef.shape[1]} columns")

df has 59 rows, and 10 columns


In [3]:
#I want to be able to see every column when doing df.head(), hence the below line of code
pd.set_option("display.max_columns", None)
dfDdef.head()

,Name,Team,Tackles,Clearances,Interceptions,Aerials Won,Blocks,Mins,Duels Won,Unnamed: 8
0,E. Valentini,1. FC Nürnberg,3,2,0,1,0,90,9,NaN
1,S. Jung,Karlsruher SC,38,64,22,13,10,2712,64,NaN
2,M. Zimmermann,Fortuna Düsseldorf,64,62,24,33,6,2221,126,NaN
3,T. Kalas,FC Schalke 04,16,95,27,62,0,1319,95,NaN
4,M. Halstenberg,Hannover 96,42,99,37,42,8,2492,110,NaN


We can already see "NaN" for an entire column, not entirely sure what the webscraper picked up that is unnamed (the number varies between filters), this'll be removed for every csv further down.

There is a possible overlapping statistic (we will find this out soon) in "Mins", which isn't necessarily a "defending" statistic, just a general one.

In [4]:
#to confirm our suspicion, lets look at another dataset
dfDdft = pd.read_csv("data/finalSCRAPEDdata/2bundesliga/2BULIdefenderDEFAULT.csv") #same dataset as above but instead of defending stats its "default" (whatever the website deems it to be)

dfDdft.head()

,Name,Team,App,Goals,Mins,XG,XA,Unnamed: 6
0,E. Valentini,1. FC Nürnberg,14,0,90,0,0,NaN
1,S. Jung,Karlsruher SC,31,2,2712,0,0,NaN
2,M. Zimmermann,Fortuna Düsseldorf,33,3,2221,0,0,NaN
3,T. Kalas,FC Schalke 04,27,1,1319,0,0,NaN
4,M. Halstenberg,Hannover 96,28,3,2492,0,0,NaN


Suspicion confirmed, we may as well check the other 3 variations of the datasets. (one of them will be changed to goalkeeper as that is the only football position with relevant statistics)

In [5]:
dfDatt = pd.read_csv("data/finalSCRAPEDdata/2bundesliga/2BULIdefenderATTACKING.csv")
dfDpas = pd.read_csv("data/finalSCRAPEDdata/2bundesliga/2BULIdefenderPASSING.csv")
dfGgoa = pd.read_csv("data/finalSCRAPEDdata/2bundesliga/2BULIgoalkeeperGOALKEEPING.csv")

dfDatt.head()

,Name,Team,Assists,Big Chances Created,Shots,Shots On Target,Goals,Mins,XG,NpxG,XA,Unnamed: 10
0,E. Valentini,1. FC Nürnberg,1,2,1.0,0,0,90,0.0,NaN,0.0,NaN
1,S. Jung,Karlsruher SC,1,3,26.0,6,2,2712,0.0,NaN,0.0,NaN
2,M. Zimmermann,Fortuna Düsseldorf,2,3,24.0,10,3,2221,0.0,NaN,0.0,NaN
3,T. Kalas,FC Schalke 04,1,1,6.0,2,1,1319,0.0,NaN,0.0,NaN
4,M. Halstenberg,Hannover 96,2,2,26.0,8,3,2492,0.0,NaN,0.0,NaN


In [6]:
dfDpas.head()

,Name,Team,Passes,Accurate Passes,Long Balls,Assists,Crosses,Big Chances Created,Mins,KeyPasses,Unnamed: 9
0,E. Valentini,1. FC Nürnberg,66,56,12,1,8,2,90,4,NaN
1,S. Jung,Karlsruher SC,1047,813,135,1,106,3,2712,21,NaN
2,M. Zimmermann,Fortuna Düsseldorf,1212,1023,69,2,58,3,2221,18,NaN
3,T. Kalas,FC Schalke 04,699,580,86,1,9,1,1319,6,NaN
4,M. Halstenberg,Hannover 96,1536,1303,222,2,37,2,2492,15,NaN


In [7]:
dfGgoa.head()

,Name,Team,Mins,Saves,Unnamed: 3
0,M. Langer,FC Schalke 04,0,0,NaN
1,T. Mickel,Hamburger SV,0,0,NaN
2,R. Himmelmann,Karlsruher SC,0,0,NaN
3,D. Heuer Fernandes,Hamburger SV,2790,77,NaN
4,M. Schuhen,Darmstadt 98,3051,103,NaN


Every overlap between the datasets can be seen in the table below:

__"unnamed" isn't mentioned in the table, as it is going to be dropped__

__Alongside that, "Name" and "Team" aren't mentioned here as they are important identifiers for the time being__

|  | ATTACKING | DEFAULT | DEFENDING | PASSING | GOALKEEPING |
|:--------:|:--------:|:--------:|:--------:|:--------:|:--------:|
| __ATTACKING__|  | Goals + Mins + XG + XA | Mins | Assists + Big Chances Created + Mins | Mins |
| __DEFAULT__ | --- |  | Mins | Mins | Mins |
| __DEFENDING__ | --- | --- |  | Mins | Mins |
| __PASSING__ | --- | --- | --- |  | Mins |
| __GOALKEEPING__ | --- | --- | --- | --- |  |  

<br>

Minutes is the most overlapped column, followed by the ATTACKING datasets having a big overlap with default and passing.

__idk if the below chapter is bullshit or not, might not move XA but maybe during cleaning make the existing csvs finalised, then make a merged final csv but have these just as backups?__
As xA is a calculation of how probable the assist was, I opted to move XA away from attacking and to the passing dataset.
Even though the goal is to make one big dataset, further work will still have statistics sorted into roughly the above categories for aiding users in useful visualisations in the final data dashboard.

###  1.1.1 Names of features/classes

I like to work with camelCase naming schemes so in this step of data exploration/cleaning, I will be renaming column titles where applicable to camelCase.

For the sake of clarity, all of this was done in one code cell per position.

In [8]:
#Initialising the rest of the position dataframes

#FORWARD
dfFatt = pd.read_csv("data/finalSCRAPEDdata/2bundesliga/2BULIforwardATTACKING.csv")
dfFdft = pd.read_csv("data/finalSCRAPEDdata/2bundesliga/2BULIforwardDEFAULT.csv")
dfFdef = pd.read_csv("data/finalSCRAPEDdata/2bundesliga/2BULIforwardDEFENDING.csv")
dfFpas = pd.read_csv("data/finalSCRAPEDdata/2bundesliga/2BULIforwardPASSING.csv")

#GOALKEEPER
dfGatt = pd.read_csv("data/finalSCRAPEDdata/2bundesliga/2BULIgoalkeeperATTACKING.csv")
dfGdft = pd.read_csv("data/finalSCRAPEDdata/2bundesliga/2BULIgoalkeeperDEFAULT.csv")
dfGdef = pd.read_csv("data/finalSCRAPEDdata/2bundesliga/2BULIgoalkeeperDEFENDING.csv")
dfGgoa = pd.read_csv("data/finalSCRAPEDdata/2bundesliga/2BULIgoalkeeperGOALKEEPING.csv") 
dfGpas = pd.read_csv("data/finalSCRAPEDdata/2bundesliga/2BULIgoalkeeperPASSING.csv")

#MIDFIELDER
dfMatt = pd.read_csv("data/finalSCRAPEDdata/2bundesliga/2BULImidfielderATTACKING.csv")
dfMdft = pd.read_csv("data/finalSCRAPEDdata/2bundesliga/2BULImidfielderDEFAULT.csv")
dfMdef = pd.read_csv("data/finalSCRAPEDdata/2bundesliga/2BULImidfielderDEFENDING.csv")
dfMpas = pd.read_csv("data/finalSCRAPEDdata/2bundesliga/2BULImidfielderPASSING.csv")

In [9]:
#arrays for renaming
attackingColumns = ["Name", "Team", "Assists", "bigChancesCreated", "Shots", "shotsOnTarget", "Goals", "Mins", "xG", "npxG", "xA", "unnamed"]
defaultColumns = ["Name", "Team", "App", "Goals", "Mins", "xG", "xA", "unnamed"]
defendingColumns = ["Name", "Team", "Tackles", "Clearances", "Interceptions", "aerialsWon", "Blocks", "Mins", "duelsWon", "unnamed"]
passingColumns = ["Name", "Team", "Passes", "accuratePasses", "longBalls", "Assists", "Crosses", "bigChancesCreated", "Mins", "keyPasses", "unnamed"]
goalkeepingColumns = ["Name", "Team", "Mins", "Saves", "unnamed"]

#attacking
for df in [dfFatt, dfGatt, dfMatt, dfDatt]:
    df.columns = attackingColumns

#default
for df in [dfFdft, dfGdft, dfMdft, dfDdft]:
    df.columns = defaultColumns

#defending
for df in [dfFdef, dfGdef, dfMdef, dfDdef]:
    df.columns = defendingColumns

#passing
for df in [dfFpas, dfGpas, dfMpas, dfDpas]:
    df.columns = passingColumns

#goalkeeping
for df in [dfGgoa]:
    df.columns = goalkeepingColumns

In [10]:
#quick check
dfFatt.head(5)
#dfFdft.head(5)
#dfDdef.head(5)
#dfMpas.head(5)
#dfGgoa.head(5)

,Name,Team,Assists,bigChancesCreated,Shots,shotsOnTarget,Goals,Mins,xG,npxG,xA,unnamed
0,A. Grimaldi,1. FC Nürnberg,3,4,57.0,22,8,1093,0.0,NaN,0.0,NaN
1,H. Nielsen,Hannover 96,3,4,39.0,10,2,1399,0.0,NaN,0.0,NaN
2,A. Voglsammer,F.C. Hansa Rostock,1,1,27.0,10,3,958,0.0,NaN,0.0,NaN
3,B. Zivzivadze,1. FC Heidenheim,3,6,56.0,18,12,1463,0.0,NaN,0.0,NaN
4,K. Karaman,FC Schalke 04,4,4,84.0,29,13,2492,0.0,NaN,0.0,NaN


In [11]:
#only unnamed is expected here
dfDdef.isnull().sum()

Name              0
Team              0
Tackles           0
Clearances        0
Interceptions     0
aerialsWon        0
Blocks            0
Mins              0
duelsWon          0
unnamed          59
dtype: int64

In [12]:
dfDdef.describe()

,Tackles,Clearances,Interceptions,aerialsWon,Blocks,Mins,duelsWon,unnamed
count,59.000000,59.000000,59.000000,59.000000,59.000000,59.000000,59.000000,0.0
mean,22.949153,48.186441,14.067797,20.677966,2.423729,1169.525424,59.406780,NaN
std,22.147954,50.685465,14.519742,24.625779,3.024061,1058.974347,54.337743,NaN
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
25%,1.000000,1.500000,0.500000,1.000000,0.000000,56.500000,3.500000,NaN
50%,17.000000,34.000000,11.000000,12.000000,1.000000,1213.000000,52.000000,NaN
75%,40.000000,79.000000,24.000000,31.500000,3.500000,2201.000000,108.000000,NaN
max,86.000000,210.000000,57.000000,108.000000,13.000000,3060.000000,204.000000,NaN


We can see that there are players with 0 minutes captured in the dataframes, they need deleted as they will have 0 in every other stat, skewing the data analysis later negatively.

This further raises the question of, how many minutes are deemed enough for being included in the data analysis? A game of football is 90 minutes and usually teams play anywhere in the 30-40 game range. Another factor impacting this decision is the lack of data, with football data becoming harder to collect, to do a complete analysis of statistics, any entry within the database is valuable.

As league coverage has been inconsistent from my webscraped source, every league will have a custom amount of Mins removed, alongside that I will conduct Ablation Studies with varying amounts of Mins present in the dataset, to investigate data quality vs data quantity.

One dataset will remove only 0 Mins players, whilst another will remove the 25% threshold from the .describe() function (in this case for 2 bundesliga that would be 56.5 Mins), __ADD MORE HERE__

The creation of abalation datasets will be later, after the master database for this league has been created.

# 1.2 Data Pre-processing

Here we will be deleting columns, adding columns and merging datasets together to make one for the league, which will later be merged into one major dataset.

### 1.2.1 Data Erasure

Attacking has the highest overlaps between the other filters, with the most coming from "default" (4) followed by "passing" (3).

Every dataset has consistent overlap with "Mins" and "unnamed"

Therefore, I will be dropping these columns:

- Attacking = bigChancesCreated, Mins
- Default = Goals, xG, xA
- Defending = Mins
- Goalkeeping = Mins
- Passing = Assists, Mins

This'll leave the structure of individual dataframes as follows:

- Attacking: Name, Team, Assists, Shots, shotsOnTarget, Goals, xG, npxG, xA
- Default: Name, Team, App, Mins
- Defending: Name, Team, Tackles, Clearances, Interceptions, aerialsWon, Blocks, duelsWon
- Goalkeeping: Name, Team, Saves
- Passing: Name, Team, Passes, accuratePasses, longBalls, Crosses, bigChancesCreated, keyPasses

In [13]:
#arrays for dropping columns
defaultDropColumns = ["Goals", "xG", "xA", "unnamed"]
defendingDropColumns = ["Mins", "unnamed"]
attackingDropColumns = ["bigChancesCreated", "Mins", "unnamed"]
passingDropColumns = ["Assists", "Mins", "unnamed"]

#Every default filter dataframe
for df in [dfDdft, dfFdft, dfGdft, dfMdft]:
    df.drop(columns=defaultDropColumns, inplace=True)

#Every defending filter dataframe
for df in [dfDdef, dfFdef, dfGdef, dfMdef]:
    df.drop(columns=defendingDropColumns, inplace=True)

#Every attacking filter dataframe
for df in [dfDatt, dfFatt, dfGatt, dfMatt]:
    df.drop(columns=attackingDropColumns, inplace=True)

#Every passing filter dataframe
for df in [dfDpas, dfFpas, dfGpas, dfMpas]:
    df.drop(columns=passingDropColumns, inplace=True)

#Every goalkeeping dataframe
dfGgoa =  dfGgoa.drop(columns=["Mins", "unnamed"])

#quick check
dfFatt.head(5)
#dfFdft.head(5)
#dfDdef.head(5)
#dfMpas.head(5)
#dfGgoa.head(5)

,Name,Team,Assists,Shots,shotsOnTarget,Goals,xG,npxG,xA
0,A. Grimaldi,1. FC Nürnberg,3,57.0,22,8,0.0,NaN,0.0
1,H. Nielsen,Hannover 96,3,39.0,10,2,0.0,NaN,0.0
2,A. Voglsammer,F.C. Hansa Rostock,1,27.0,10,3,0.0,NaN,0.0
3,B. Zivzivadze,1. FC Heidenheim,3,56.0,18,12,0.0,NaN,0.0
4,K. Karaman,FC Schalke 04,4,84.0,29,13,0.0,NaN,0.0


__This varies in every datacleaning file, as the identifier will have to be picked up from the final id of the previous file (so next alphabetically is the bundesliga and that id will have to start from whatever the final id number is from here)__

Before merging, we add a quick unique identifier to every player to make the merge simpler

In [14]:
#adding unique identifier, from manually looking at the data I can see that the webscraper has picked up on the data in the same order every time
#we  index everything automatically at the defenders first
dfDdft["ID"] = dfDdft.index
dfDdef["ID"] = dfDdef.index
dfDatt["ID"] = dfDatt.index
dfDpas["ID"] = dfDpas.index #should end at id no 58
#dfDdft.head(300) - to check final id number

#next position alphabetically is forwards, so we find the max value for the defenders id
defFinalID = dfDdft["ID"].max()

#forwards id
dfFdft["ID"] = range(defFinalID + 1, defFinalID + 1 + len(dfFdft))
dfFdef["ID"] = range(defFinalID + 1, defFinalID + 1 + len(dfFdef))
dfFatt["ID"] = range(defFinalID + 1, defFinalID + 1 + len(dfFatt))
dfFpas["ID"] = range(defFinalID + 1, defFinalID + 1 + len(dfFpas)) #should end at id no. 96
#dfFdft.head(300) - to check final id number

#repeat of finding the forwards id this time
forFinalID = dfFdft["ID"].max()

#goalkeepers id
dfGdft["ID"] = range(forFinalID +1, forFinalID +1 +len(dfGdft))
dfGdef["ID"] = range(forFinalID +1, forFinalID +1 +len(dfGdef))
dfGatt["ID"] = range(forFinalID +1, forFinalID +1 +len(dfGatt))
dfGpas["ID"] = range(forFinalID +1, forFinalID +1 +len(dfGpas))
dfGgoa["ID"] = range(forFinalID +1, forFinalID +1 +len(dfGgoa)) #should end at id no. 118
#dfGdft.head(300) - to check final id number

#repeat of finding the goalkeepers id this time
goaFinalId = dfGdft["ID"].max()

#midfielders id
dfMdft["ID"] = range(goaFinalId + 1, goaFinalId + 1 +len(dfMdft))
dfMdef["ID"] = range(goaFinalId + 1, goaFinalId + 1 +len(dfMdef))
dfMatt["ID"] = range(goaFinalId + 1, goaFinalId + 1 +len(dfMatt))
dfMpas["ID"] = range(goaFinalId + 1, goaFinalId + 1 +len(dfMpas)) #should end at id no. 169
dfMdft.head(300) #to check final id number

,Name,Team,App,Mins,ID
0,D. Soares,1. FC Nürnberg,31,1514,119
1,A. Geipl,SSV Jahn Regensburg,31,1744,120
2,R. Krausse,SV Sandhausen,31,1860,121
3,J. Meffert,Holstein Kiel,29,2375,122
4,JGJ. Günther-Schmidt,Erzgebirge Aue,2,0,123
5,D. Kinsombi,SC Paderborn 07,32,1062,124
6,P. Förster,Karlsruher SC,22,1456,125
7,M. Wanitzek,Karlsruher SC,34,3060,126
8,J. Bachmann,FC Schalke 04,25,1646,127
9,N. Rapp,Karlsruher SC,29,2157,128


#### Further Data Erasure

Now that we have the player identifiers, to ensure the merging goes smoothly, we drop "Name" and "Team" from every dataframe but the default ones

In [15]:
#array for dropping name and team
nameAndTeamDropColumns = ["Name", "Team"]

#Every defending filter dataframe
for df in [dfDdef, dfFdef, dfGdef, dfMdef]:
    df.drop(columns=nameAndTeamDropColumns, inplace=True)

#Every attacking filter dataframe
for df in [dfDatt, dfFatt, dfGatt, dfMatt]:
    df.drop(columns=nameAndTeamDropColumns, inplace=True)

#Every passing filter dataframe
for df in [dfDpas, dfFpas, dfGpas, dfMpas]:
    df.drop(columns=nameAndTeamDropColumns, inplace=True)

#Every goalkeeping dataframe
dfGgoa =  dfGgoa.drop(columns=["Name", "Team"])

#### Dataframe Merging

Now that we have sorted out our unique identifier, we can merge every positional database together to create master-positional databases (for example, 2BULIdefender.csv)

In [16]:
#merging defenders
defenderDFs = [dfDdft, dfDatt, dfDdef, dfDpas]

defendersMerged = reduce(lambda left,right: pd.merge(left,right, on="ID", how="outer"), defenderDFs)

#merging forwards
forwardDFs = [dfFdft, dfFatt, dfFdef, dfFpas]

forwardsMerged = reduce(lambda left,right: pd.merge(left,right, on="ID", how="outer"), forwardDFs)

#merging goalkeepers
goalkeeperDFs = [dfGdft, dfGatt, dfGdef, dfGpas, dfGgoa]

goalkeepersMerged = reduce(lambda left,right: pd.merge(left,right, on="ID", how="outer"), goalkeeperDFs)

#merging midfielders
midfielderDFs = [dfMdft, dfMatt, dfMdef, dfMpas]

midfieldersMerged = reduce(lambda left,right: pd.merge(left,right, on="ID", how="outer"), midfielderDFs)

Now that all of the filter datasets are merged under 1 per position, we can add a "position" label to later be able to distinguish player positions in the data dashboard.

In [17]:
#adding defender column
#creating label
defenderPosition = "Defender"
#adding label to df
defendersMerged["Position"] = defenderPosition

#adding forward column
#creating label
forwardPosition = "Forward"
#adding label to df
forwardsMerged["Position"] = forwardPosition

#adding goalkeeper column
#creating label
goalkeeperPosition = "Goalkeeper"
#adding label to df
goalkeepersMerged["Position"] = goalkeeperPosition

#adding midfielder column
#creating label
midfielderPosition = "Midfielder"
#adding label to df
midfieldersMerged["Position"] = midfielderPosition

In [18]:
#Just as a precaution, we can save the merged datasets under separate csvs if needed for some reason

defendersMerged.to_csv("data/finalSCRAPEDdata/2bundesliga/2BULIdefender.csv", index=None)
forwardsMerged.to_csv("data/finalSCRAPEDdata/2bundesliga/2BULIforward.csv", index=None)
goalkeepersMerged.to_csv("data/finalSCRAPEDdata/2bundesliga/2BULIgoalkeeper.csv", index=None)
midfieldersMerged.to_csv("data/finalSCRAPEDdata/2bundesliga/2BULImidfielder.csv", index=None)

Now that we have a unique identifier for every player and distinguished which position they play, we can create the master league database by merging all the positional datasets together.

In [19]:
#merging everything
positionDFs = [defendersMerged, forwardsMerged, goalkeepersMerged, midfieldersMerged]

finalData = pd.concat(positionDFs, ignore_index=True)

finalData.head(300)

,Name,Team,App,Mins,ID,Assists,Shots,shotsOnTarget,Goals,xG,npxG,xA,Tackles,Clearances,Interceptions,aerialsWon,Blocks,duelsWon,Passes,accuratePasses,longBalls,Crosses,bigChancesCreated,keyPasses,Position,Saves
0,E. Valentini,1. FC Nürnberg,14,90,0,1,1.0,0,0,0.0,NaN,0.0,3,2,0,1,0,9,66,56,12,8,2,4,Defender,NaN
1,S. Jung,Karlsruher SC,31,2712,1,1,26.0,6,2,0.0,NaN,0.0,38,64,22,13,10,64,1047,813,135,106,3,21,Defender,NaN
2,M. Zimmermann,Fortuna Düsseldorf,33,2221,2,2,24.0,10,3,0.0,NaN,0.0,64,62,24,33,6,126,1212,1023,69,58,3,18,Defender,NaN
3,T. Kalas,FC Schalke 04,27,1319,3,1,6.0,2,1,0.0,NaN,0.0,16,95,27,62,0,95,699,580,86,9,1,6,Defender,NaN
4,M. Halstenberg,Hannover 96,28,2492,4,2,26.0,8,3,0.0,NaN,0.0,42,99,37,42,8,110,1536,1303,222,37,2,15,Defender,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
165,H. Chakroun,Hannover 96,8,13,165,0,NaN,0,0,0.0,NaN,0.0,0,0,0,0,0,0,2,0,1,0,0,0,Midfielder,NaN
166,S. Suso,Fortuna Düsseldorf,4,1,166,0,NaN,0,0,0.0,NaN,0.0,0,0,0,0,0,1,4,4,0,0,1,1,Midfielder,NaN
167,L. Ben Farhat,Karlsruher SC,14,452,167,0,14.0,8,3,0.0,NaN,0.0,5,1,3,7,3,26,66,45,3,1,3,6,Midfielder,NaN
168,M. Kritzer,Karlsruher SC,1,0,168,0,NaN,0,0,0.0,NaN,0.0,0,0,0,0,0,0,0,0,0,0,0,0,Midfielder,NaN


### 1.2.2 Further Data Cleaning

Now that we have our final merged dataset, we can continue to clean it further on a wider scale.

First course of action is handling missing values (like midfielders having NaN shots blocked, as that is a goalkeeper exclusive stat 99.99% of the time)

In [20]:
numberColumns = finalData.select_dtypes(include=["float64", "int64"]).columns

finalData[numberColumns] = finalData[numberColumns].fillna(0)

finalData.head(-5)

,Name,Team,App,Mins,ID,Assists,Shots,shotsOnTarget,Goals,xG,npxG,xA,Tackles,Clearances,Interceptions,aerialsWon,Blocks,duelsWon,Passes,accuratePasses,longBalls,Crosses,bigChancesCreated,keyPasses,Position,Saves
0,E. Valentini,1. FC Nürnberg,14,90,0,1,1.0,0,0,0.0,0.0,0.0,3,2,0,1,0,9,66,56,12,8,2,4,Defender,0.0
1,S. Jung,Karlsruher SC,31,2712,1,1,26.0,6,2,0.0,0.0,0.0,38,64,22,13,10,64,1047,813,135,106,3,21,Defender,0.0
2,M. Zimmermann,Fortuna Düsseldorf,33,2221,2,2,24.0,10,3,0.0,0.0,0.0,64,62,24,33,6,126,1212,1023,69,58,3,18,Defender,0.0
3,T. Kalas,FC Schalke 04,27,1319,3,1,6.0,2,1,0.0,0.0,0.0,16,95,27,62,0,95,699,580,86,9,1,6,Defender,0.0
4,M. Halstenberg,Hannover 96,28,2492,4,2,26.0,8,3,0.0,0.0,0.0,42,99,37,42,8,110,1536,1303,222,37,2,15,Defender,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160,N. Baier,Darmstadt 98,2,0,160,0,0.0,0,0,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,Midfielder,0.0
161,N. Aséko,Hannover 96,5,22,161,0,0.0,0,0,0.0,0.0,0.0,1,0,0,0,0,1,2,2,0,0,0,0,Midfielder,0.0
162,K. Affo,Fortuna Düsseldorf,11,8,162,0,0.0,0,0,0.0,0.0,0.0,1,0,0,0,0,2,2,1,1,1,0,0,Midfielder,0.0
163,TJT. Janisch,1. FC Nürnberg,20,1112,163,0,3.0,0,0,0.0,0.0,0.0,35,23,10,2,0,74,536,434,50,5,2,4,Midfielder,0.0


Next, we want to clean up the column order slightly, for bookkeeping.

We specifically move the "ID" column to the furthest left, and "Position" to the second-most left column, as .head() already shows a unique identifier as the first row (although that is not a part of the actual csv itself), it just makes sense to match it and adapt it for the csv, as for position, similar logic follows, it is an important identifier for the players identity.

In [21]:
#id
finalData.insert(0, "ID", finalData.pop("ID"))
#position
finalData.insert(1, "Position", finalData.pop("Position"))

finalData.head(-5)

,ID,Position,Name,Team,App,Mins,Assists,Shots,shotsOnTarget,Goals,xG,npxG,xA,Tackles,Clearances,Interceptions,aerialsWon,Blocks,duelsWon,Passes,accuratePasses,longBalls,Crosses,bigChancesCreated,keyPasses,Saves
0,0,Defender,E. Valentini,1. FC Nürnberg,14,90,1,1.0,0,0,0.0,0.0,0.0,3,2,0,1,0,9,66,56,12,8,2,4,0.0
1,1,Defender,S. Jung,Karlsruher SC,31,2712,1,26.0,6,2,0.0,0.0,0.0,38,64,22,13,10,64,1047,813,135,106,3,21,0.0
2,2,Defender,M. Zimmermann,Fortuna Düsseldorf,33,2221,2,24.0,10,3,0.0,0.0,0.0,64,62,24,33,6,126,1212,1023,69,58,3,18,0.0
3,3,Defender,T. Kalas,FC Schalke 04,27,1319,1,6.0,2,1,0.0,0.0,0.0,16,95,27,62,0,95,699,580,86,9,1,6,0.0
4,4,Defender,M. Halstenberg,Hannover 96,28,2492,2,26.0,8,3,0.0,0.0,0.0,42,99,37,42,8,110,1536,1303,222,37,2,15,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160,160,Midfielder,N. Baier,Darmstadt 98,2,0,0,0.0,0,0,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0.0
161,161,Midfielder,N. Aséko,Hannover 96,5,22,0,0.0,0,0,0.0,0.0,0.0,1,0,0,0,0,1,2,2,0,0,0,0,0.0
162,162,Midfielder,K. Affo,Fortuna Düsseldorf,11,8,0,0.0,0,0,0.0,0.0,0.0,1,0,0,0,0,2,2,1,1,1,0,0,0.0
163,163,Midfielder,TJT. Janisch,1. FC Nürnberg,20,1112,0,3.0,0,0,0.0,0.0,0.0,35,23,10,2,0,74,536,434,50,5,2,4,0.0


Now more data erasure is potentially due, from my research into footballing data, and as reported multiple times throughout my report, the loss of consistent and deep data sources has been plaguing me, and the greater football analysis community with mainly the situation around the website "FBRef"

A quick reminder is that in Feburary 2026, FBRef were ordered by their data-supplier to take away "advanced statistics" (xG, npxG and xA in our context, that affect us directly), hence why you can see above, that xG, npxG and xA all have 0 throughout, my data-source was directly impacted by the FBRef situation, as has any other source I looked at.

We will need to remove this for any league that doesn't cover these advanced statistics, which is most but not all from what I noted when doing initial manual data cleaning.

Just to confirm the above, let's inspect the merged dataframe closer.

In [22]:
finalData.describe()

,ID,App,Mins,Assists,Shots,shotsOnTarget,Goals,xG,npxG,xA,Tackles,Clearances,Interceptions,aerialsWon,Blocks,duelsWon,Passes,accuratePasses,longBalls,Crosses,bigChancesCreated,keyPasses,Saves
count,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.0,170.0,170.0,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000
mean,84.500000,20.682353,1090.941176,1.205882,14.876471,5.170588,1.776471,0.0,0.0,0.0,17.258824,25.452941,8.941176,15.911765,3.864706,51.470588,465.835294,380.264706,59.717647,22.935294,2.047059,11.729412,3.423529
std,49.218899,11.603623,1009.588426,2.229169,20.229805,8.407539,3.380685,0.0,0.0,0.0,19.554598,36.491559,11.805979,20.412851,5.410864,50.745586,510.766679,436.035916,95.682675,47.583890,3.393017,17.287947,16.886588
min,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,42.250000,10.250000,51.750000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,14.500000,11.000000,1.000000,0.000000,0.000000,0.000000,0.000000
50%,84.500000,25.000000,890.000000,0.000000,8.500000,2.000000,0.000000,0.0,0.0,0.0,9.500000,10.500000,3.000000,8.000000,2.000000,35.000000,298.500000,217.500000,18.000000,5.500000,1.000000,6.000000,0.000000
75%,126.750000,31.000000,2016.000000,1.750000,19.750000,6.000000,2.000000,0.0,0.0,0.0,28.000000,34.750000,14.000000,24.750000,6.000000,89.000000,750.750000,602.000000,83.750000,18.750000,2.750000,15.750000,0.000000
max,169.000000,35.000000,3060.000000,12.000000,93.000000,40.000000,16.000000,0.0,0.0,0.0,86.000000,210.000000,57.000000,108.000000,27.000000,204.000000,1984.000000,1666.000000,644.000000,258.000000,21.000000,82.000000,127.000000


__This section varies in these files: dataCleaningPREMIERLEAGUE.ipynb,__

Our suspicions are confirmed, so we will now unfortunately drop these fields.

In [23]:
finalData = finalData.drop(columns=["xG", "npxG", "xA"])

### Adding Per90 stats

However, we can also add additional data fields that weren't initially calculated by my data source, with the importance of "per90" statistics explained in my report, we can add them into this database to further the depth of our data.

We can also quickly add a goals+assists column

In [24]:
#Adding per90 stats - grouped for visual reason
#attacking orientated stats
#goals per90
finalData["goalsPer90"] = finalData["Goals"] / finalData["Mins"] * 90
#assists per90
finalData["assistsPer90"] = finalData["Assists"] / finalData["Mins"] * 90
#goals+assists (to make per90)
finalData["g+a"] = finalData["Assists"] + finalData["Goals"]
#goals+assists per 90
finalData["g+aPer90"] = finalData["g+a"] / finalData["Mins"] * 90
#shots per 90
finalData["shotsPer90"] = finalData["Shots"] / finalData["Mins"] * 90
#shotsOnTarget per 90
finalData["shotsOnTargetPer90"] = finalData["shotsOnTarget"] / finalData["Mins"] * 90
#duelsWon per 90
finalData["duelsWonPer90"] = finalData["duelsWon"] / finalData["Mins"] * 90

#passing orientated stats
#passes per 90
finalData["passesPer90"] = finalData["Passes"] / finalData["Mins"] * 90
#accuratePasses per 90
finalData["accuratePassesPer90"] = finalData["accuratePasses"] / finalData["Mins"] * 90
#longBalls per 90
finalData["longBallsPer90"] = finalData["longBalls"] / finalData["Mins"] * 90
#crosses per 90
finalData["crossesPer90"] = finalData["Crosses"] / finalData["Mins"] * 90
#bigChancesCreated per 90
finalData["bigChancesCreatedPer90"] = finalData["bigChancesCreated"] / finalData["Mins"] * 90
#keyPasses per 90
finalData["keyPassesPer90"] = finalData["keyPasses"] / finalData["Mins"] * 90
 
#defending orientated stats
#tackles per 90
finalData["tacklesPer90"] = finalData["Tackles"] / finalData["Mins"] * 90
#clearances per 90
finalData["clearancesPer90"] = finalData["Clearances"] / finalData["Mins"] * 90
#interceptions per 90
finalData["interceptionsPer90"] = finalData["Interceptions"] / finalData["Mins"] * 90
#aerialsWon per 90
finalData["aerialsWonPer90"] = finalData["aerialsWon"] / finalData["Mins"] * 90
#blocks per 90
finalData["blocksPer90"] = finalData["Blocks"] / finalData["Mins"] * 90
#savse per 90 - only goalkeeping stat available to us
finalData["savesPer90"] = finalData["Saves"] / finalData["Mins"] * 90

finalData.head(-5)

,ID,Position,Name,Team,App,Mins,Assists,Shots,shotsOnTarget,Goals,Tackles,Clearances,Interceptions,aerialsWon,Blocks,duelsWon,Passes,accuratePasses,longBalls,Crosses,bigChancesCreated,keyPasses,Saves,goalsPer90,assistsPer90,g+a,g+aPer90,shotsPer90,shotsOnTargetPer90,duelsWonPer90,passesPer90,accuratePassesPer90,longBallsPer90,crossesPer90,bigChancesCreatedPer90,keyPassesPer90,tacklesPer90,clearancesPer90,interceptionsPer90,aerialsWonPer90,blocksPer90,savesPer90
0,0,Defender,E. Valentini,1. FC Nürnberg,14,90,1,1.0,0,0,3,2,0,1,0,9,66,56,12,8,2,4,0.0,0.000000,1.000000,1,1.000000,1.000000,0.000000,9.000000,66.000000,56.000000,12.000000,8.000000,2.000000,4.000000,3.000000,2.000000,0.000000,1.000000,0.000000,0.0
1,1,Defender,S. Jung,Karlsruher SC,31,2712,1,26.0,6,2,38,64,22,13,10,64,1047,813,135,106,3,21,0.0,0.066372,0.033186,3,0.099558,0.862832,0.199115,2.123894,34.745575,26.980088,4.480088,3.517699,0.099558,0.696903,1.261062,2.123894,0.730088,0.431416,0.331858,0.0
2,2,Defender,M. Zimmermann,Fortuna Düsseldorf,33,2221,2,24.0,10,3,64,62,24,33,6,126,1212,1023,69,58,3,18,0.0,0.121567,0.081045,5,0.202611,0.972535,0.405223,5.105808,49.113012,41.454300,2.796038,2.350293,0.121567,0.729401,2.593426,2.512382,0.972535,1.337235,0.243134,0.0
3,3,Defender,T. Kalas,FC Schalke 04,27,1319,1,6.0,2,1,16,95,27,62,0,95,699,580,86,9,1,6,0.0,0.068234,0.068234,2,0.136467,0.409401,0.136467,6.482183,47.695224,39.575436,5.868082,0.614102,0.068234,0.409401,1.091736,6.482183,1.842305,4.230478,0.000000,0.0
4,4,Defender,M. Halstenberg,Hannover 96,28,2492,2,26.0,8,3,42,99,37,42,8,110,1536,1303,222,37,2,15,0.0,0.108347,0.072231,5,0.180578,0.939005,0.288925,3.972713,55.473515,47.058587,8.017657,1.336276,0.072231,0.541734,1.516854,3.575441,1.336276,1.516854,0.288925,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160,160,Midfielder,N. Baier,Darmstadt 98,2,0,0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
161,161,Midfielder,N. Aséko,Hannover 96,5,22,0,0.0,0,0,1,0,0,0,0,1,2,2,0,0,0,0,0.0,0.000000,0.000000,0,0.000000,0.000000,0.000000,4.090909,8.181818,8.181818,0.000000,0.000000,0.000000,0.000000,4.090909,0.000000,0.000000,0.000000,0.000000,0.0
162,162,Midfielder,K. Affo,Fortuna Düsseldorf,11,8,0,0.0,0,0,1,0,0,0,0,2,2,1,1,1,0,0,0.0,0.000000,0.000000,0,0.000000,0.000000,0.000000,22.500000,22.500000,11.250000,11.250000,11.250000,0.000000,0.000000,11.250000,0.000000,0.000000,0.000000,0.000000,0.0
163,163,Midfielder,TJT. Janisch,1. FC Nürnberg,20,1112,0,3.0,0,0,35,23,10,2,0,74,536,434,50,5,2,4,0.0,0.000000,0.000000,0,0.000000,0.242806,0.000000,5.989209,43.381295,35.125899,4.046763,0.404676,0.161871,0.323741,2.832734,1.861511,0.809353,0.161871,0.000000,0.0


We can once again see that some players have NaN in fields, we will clean it in the same way as we did previously in this file, however, the extra decimal spaces are unnecessary.

In already existing softwares expected statistics (xG, xA etc.) and per90 statistics are measured up to 2 decimal places, so we will adjust that too.

In [25]:
#fill NaN rows with 0 (reused code)
numberColumns = finalData.select_dtypes(include=["float64", "int64"]).columns
finalData[numberColumns] = finalData[numberColumns].fillna(0)

#round decimals to 2
finalData = finalData.round(decimals=2)

finalData.head(-5)

,ID,Position,Name,Team,App,Mins,Assists,Shots,shotsOnTarget,Goals,Tackles,Clearances,Interceptions,aerialsWon,Blocks,duelsWon,Passes,accuratePasses,longBalls,Crosses,bigChancesCreated,keyPasses,Saves,goalsPer90,assistsPer90,g+a,g+aPer90,shotsPer90,shotsOnTargetPer90,duelsWonPer90,passesPer90,accuratePassesPer90,longBallsPer90,crossesPer90,bigChancesCreatedPer90,keyPassesPer90,tacklesPer90,clearancesPer90,interceptionsPer90,aerialsWonPer90,blocksPer90,savesPer90
0,0,Defender,E. Valentini,1. FC Nürnberg,14,90,1,1.0,0,0,3,2,0,1,0,9,66,56,12,8,2,4,0.0,0.00,1.00,1,1.00,1.00,0.00,9.00,66.00,56.00,12.00,8.00,2.00,4.00,3.00,2.00,0.00,1.00,0.00,0.0
1,1,Defender,S. Jung,Karlsruher SC,31,2712,1,26.0,6,2,38,64,22,13,10,64,1047,813,135,106,3,21,0.0,0.07,0.03,3,0.10,0.86,0.20,2.12,34.75,26.98,4.48,3.52,0.10,0.70,1.26,2.12,0.73,0.43,0.33,0.0
2,2,Defender,M. Zimmermann,Fortuna Düsseldorf,33,2221,2,24.0,10,3,64,62,24,33,6,126,1212,1023,69,58,3,18,0.0,0.12,0.08,5,0.20,0.97,0.41,5.11,49.11,41.45,2.80,2.35,0.12,0.73,2.59,2.51,0.97,1.34,0.24,0.0
3,3,Defender,T. Kalas,FC Schalke 04,27,1319,1,6.0,2,1,16,95,27,62,0,95,699,580,86,9,1,6,0.0,0.07,0.07,2,0.14,0.41,0.14,6.48,47.70,39.58,5.87,0.61,0.07,0.41,1.09,6.48,1.84,4.23,0.00,0.0
4,4,Defender,M. Halstenberg,Hannover 96,28,2492,2,26.0,8,3,42,99,37,42,8,110,1536,1303,222,37,2,15,0.0,0.11,0.07,5,0.18,0.94,0.29,3.97,55.47,47.06,8.02,1.34,0.07,0.54,1.52,3.58,1.34,1.52,0.29,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160,160,Midfielder,N. Baier,Darmstadt 98,2,0,0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.00,0.00,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0
161,161,Midfielder,N. Aséko,Hannover 96,5,22,0,0.0,0,0,1,0,0,0,0,1,2,2,0,0,0,0,0.0,0.00,0.00,0,0.00,0.00,0.00,4.09,8.18,8.18,0.00,0.00,0.00,0.00,4.09,0.00,0.00,0.00,0.00,0.0
162,162,Midfielder,K. Affo,Fortuna Düsseldorf,11,8,0,0.0,0,0,1,0,0,0,0,2,2,1,1,1,0,0,0.0,0.00,0.00,0,0.00,0.00,0.00,22.50,22.50,11.25,11.25,11.25,0.00,0.00,11.25,0.00,0.00,0.00,0.00,0.0
163,163,Midfielder,TJT. Janisch,1. FC Nürnberg,20,1112,0,3.0,0,0,35,23,10,2,0,74,536,434,50,5,2,4,0.0,0.00,0.00,0,0.00,0.24,0.00,5.99,43.38,35.13,4.05,0.40,0.16,0.32,2.83,1.86,0.81,0.16,0.00,0.0


Now we will quickly add a league column and then move it to be next to position

In [26]:
#label
leagueLabel = "2. Bundesliga"

#adding label to df
finalData["League"] = leagueLabel

In [27]:
#moving league
finalData.insert(2, "League", finalData.pop("League"))

finalData.head(-5)

,ID,Position,League,Name,Team,App,Mins,Assists,Shots,shotsOnTarget,Goals,Tackles,Clearances,Interceptions,aerialsWon,Blocks,duelsWon,Passes,accuratePasses,longBalls,Crosses,bigChancesCreated,keyPasses,Saves,goalsPer90,assistsPer90,g+a,g+aPer90,shotsPer90,shotsOnTargetPer90,duelsWonPer90,passesPer90,accuratePassesPer90,longBallsPer90,crossesPer90,bigChancesCreatedPer90,keyPassesPer90,tacklesPer90,clearancesPer90,interceptionsPer90,aerialsWonPer90,blocksPer90,savesPer90
0,0,Defender,2. Bundesliga,E. Valentini,1. FC Nürnberg,14,90,1,1.0,0,0,3,2,0,1,0,9,66,56,12,8,2,4,0.0,0.00,1.00,1,1.00,1.00,0.00,9.00,66.00,56.00,12.00,8.00,2.00,4.00,3.00,2.00,0.00,1.00,0.00,0.0
1,1,Defender,2. Bundesliga,S. Jung,Karlsruher SC,31,2712,1,26.0,6,2,38,64,22,13,10,64,1047,813,135,106,3,21,0.0,0.07,0.03,3,0.10,0.86,0.20,2.12,34.75,26.98,4.48,3.52,0.10,0.70,1.26,2.12,0.73,0.43,0.33,0.0
2,2,Defender,2. Bundesliga,M. Zimmermann,Fortuna Düsseldorf,33,2221,2,24.0,10,3,64,62,24,33,6,126,1212,1023,69,58,3,18,0.0,0.12,0.08,5,0.20,0.97,0.41,5.11,49.11,41.45,2.80,2.35,0.12,0.73,2.59,2.51,0.97,1.34,0.24,0.0
3,3,Defender,2. Bundesliga,T. Kalas,FC Schalke 04,27,1319,1,6.0,2,1,16,95,27,62,0,95,699,580,86,9,1,6,0.0,0.07,0.07,2,0.14,0.41,0.14,6.48,47.70,39.58,5.87,0.61,0.07,0.41,1.09,6.48,1.84,4.23,0.00,0.0
4,4,Defender,2. Bundesliga,M. Halstenberg,Hannover 96,28,2492,2,26.0,8,3,42,99,37,42,8,110,1536,1303,222,37,2,15,0.0,0.11,0.07,5,0.18,0.94,0.29,3.97,55.47,47.06,8.02,1.34,0.07,0.54,1.52,3.58,1.34,1.52,0.29,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160,160,Midfielder,2. Bundesliga,N. Baier,Darmstadt 98,2,0,0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.00,0.00,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0
161,161,Midfielder,2. Bundesliga,N. Aséko,Hannover 96,5,22,0,0.0,0,0,1,0,0,0,0,1,2,2,0,0,0,0,0.0,0.00,0.00,0,0.00,0.00,0.00,4.09,8.18,8.18,0.00,0.00,0.00,0.00,4.09,0.00,0.00,0.00,0.00,0.0
162,162,Midfielder,2. Bundesliga,K. Affo,Fortuna Düsseldorf,11,8,0,0.0,0,0,1,0,0,0,0,2,2,1,1,1,0,0,0.0,0.00,0.00,0,0.00,0.00,0.00,22.50,22.50,11.25,11.25,11.25,0.00,0.00,11.25,0.00,0.00,0.00,0.00,0.0
163,163,Midfielder,2. Bundesliga,TJT. Janisch,1. FC Nürnberg,20,1112,0,3.0,0,0,35,23,10,2,0,74,536,434,50,5,2,4,0.0,0.00,0.00,0,0.00,0.24,0.00,5.99,43.38,35.13,4.05,0.40,0.16,0.32,2.83,1.86,0.81,0.16,0.00,0.0


In [28]:
#complete csv
finalData.to_csv("data/finalSCRAPEDdata/masterDatabases/2BULIcomplete.csv")

# 2 Ablation Studies Datasets

In this section we will split the master dataframe into separate database files, to allow for future ablation studies to go smoothly.

here are the criteria for all variations:

- 2BULIcomplete90 = Any player that has played less than 90 minutes is removed from the dataset
- 2BULIcomplete25 = Any player that has less than 25% of the minutes from the .describe() function is removed

### 2.1 90 Minute Dataset Creation

Within this section we'll create the first dataset

In [29]:
#first lets save the 90minutes  dataframe as a separate one

finalData90 = finalData

In [30]:
#Next we will move onto data erasure within the given context of <90 minutes played
finalData90 = finalData90.drop(finalData90[finalData90["Mins"] < 90].index)

In [31]:
finalData90.describe()

,ID,App,Mins,Assists,Shots,shotsOnTarget,Goals,Tackles,Clearances,Interceptions,aerialsWon,Blocks,duelsWon,Passes,accuratePasses,longBalls,Crosses,bigChancesCreated,keyPasses,Saves,goalsPer90,assistsPer90,g+a,g+aPer90,shotsPer90,shotsOnTargetPer90,duelsWonPer90,passesPer90,accuratePassesPer90,longBallsPer90,crossesPer90,bigChancesCreatedPer90,keyPassesPer90,tacklesPer90,clearancesPer90,interceptionsPer90,aerialsWonPer90,blocksPer90,savesPer90
count,124.000000,124.000000,124.000000,124.000000,124.000000,124.000000,124.000000,124.000000,124.000000,124.000000,124.000000,124.000000,124.000000,124.000000,124.000000,124.000000,124.000000,124.000000,124.000000,124.000000,124.000000,124.000000,124.000000,124.000000,124.000000,124.000000,124.000000,124.000000,124.000000,124.000000,124.000000,124.000000,124.000000,124.000000,124.000000,124.000000,124.000000,124.000000,124.000000
mean,80.620968,26.112903,1491.403226,1.653226,20.306452,7.064516,2.427419,23.580645,34.879032,12.241935,21.733871,5.266129,70.266129,637.467742,520.459677,81.750000,31.419355,2.798387,16.048387,4.693548,0.133710,0.100403,4.080645,0.234435,1.259113,0.409355,4.636935,37.419113,30.099113,4.784435,1.671048,0.175323,0.940323,1.536210,2.107339,0.734032,1.461452,0.336210,0.237177
std,50.231269,7.509571,895.998321,2.466176,21.264566,9.149830,3.757212,19.399982,38.711691,12.284150,21.119524,5.734093,47.129489,498.684129,433.587575,103.778894,53.318051,3.702295,18.469423,19.641576,0.205839,0.151917,5.376245,0.269987,0.984303,0.442489,2.000112,14.663948,13.920803,4.620931,2.023644,0.240800,0.748315,0.950569,1.686973,0.573160,1.386418,0.301159,0.845964
min,0.000000,4.000000,90.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,29.000000,17.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.090000,10.740000,6.920000,0.000000,0.000000,0.000000,0.000000,0.000000,0.190000,0.000000,0.000000,0.000000,0.000000
25%,32.750000,22.000000,647.750000,0.000000,4.750000,1.000000,0.000000,8.000000,8.000000,3.000000,5.000000,1.000000,30.000000,231.000000,172.250000,14.750000,3.000000,0.000000,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.580000,0.090000,3.702500,26.102500,18.230000,1.622500,0.277500,0.000000,0.337500,0.877500,0.805000,0.235000,0.515000,0.110000,0.000000
50%,79.500000,29.000000,1430.000000,1.000000,14.000000,4.000000,1.000000,18.000000,19.000000,9.000000,14.000000,3.000000,66.000000,521.000000,408.500000,41.500000,9.500000,2.000000,10.000000,0.000000,0.060000,0.055000,2.000000,0.155000,0.975000,0.275000,4.695000,35.045000,27.415000,3.385000,0.815000,0.110000,0.840000,1.470000,1.640000,0.650000,1.125000,0.275000,0.000000
75%,128.250000,32.000000,2280.500000,2.000000,26.250000,8.000000,3.000000,37.250000,48.500000,16.750000,31.000000,8.000000,108.250000,863.750000,713.250000,113.250000,32.250000,4.000000,20.000000,0.000000,0.162500,0.150000,5.000000,0.335000,1.902500,0.562500,5.717500,48.810000,41.467500,6.347500,2.355000,0.252500,1.340000,2.070000,2.792500,1.070000,1.772500,0.482500,0.000000
max,167.000000,35.000000,3060.000000,12.000000,93.000000,40.000000,16.000000,86.000000,210.000000,57.000000,108.000000,27.000000,204.000000,1984.000000,1666.000000,644.000000,258.000000,21.000000,82.000000,127.000000,1.330000,1.000000,24.000000,1.460000,4.690000,2.130000,11.250000,88.040000,77.280000,25.000000,9.500000,2.000000,4.000000,5.010000,7.860000,2.580000,9.640000,1.750000,4.570000


In [32]:
#Now that that has successfully gone through, we can create our next database

finalData90.to_csv("data/finalSCRAPEDdata/masterDatabases/2BULIcomplete90.csv")

### 2.2 25% Minute Dataset Creation

__This part slightly varies in every dataset__

Within this section we'll create the second dataset

In [33]:
#we'll do  the same as before, creating a separate dataframe

finalData25 = finalData

In [34]:
#first we need to find the threshold of minutes to remove
finalData25.describe()

,ID,App,Mins,Assists,Shots,shotsOnTarget,Goals,Tackles,Clearances,Interceptions,aerialsWon,Blocks,duelsWon,Passes,accuratePasses,longBalls,Crosses,bigChancesCreated,keyPasses,Saves,goalsPer90,assistsPer90,g+a,g+aPer90,shotsPer90,shotsOnTargetPer90,duelsWonPer90,passesPer90,accuratePassesPer90,longBallsPer90,crossesPer90,bigChancesCreatedPer90,keyPassesPer90,tacklesPer90,clearancesPer90,interceptionsPer90,aerialsWonPer90,blocksPer90,savesPer90
count,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000
mean,84.500000,20.682353,1090.941176,1.205882,14.876471,5.170588,1.776471,17.258824,25.452941,8.941176,15.911765,3.864706,51.470588,465.835294,380.264706,59.717647,22.935294,2.047059,11.729412,3.423529,0.104471,0.073235,2.982353,0.177941,1.021529,0.320059,5.830059,34.725235,27.821882,3.729882,1.302118,0.657294,1.255353,2.071235,2.073471,0.624588,1.696706,0.279588,0.173000
std,49.218899,11.603623,1009.588426,2.229169,20.229805,8.407539,3.380685,19.554598,36.491559,11.805979,20.412851,5.410864,50.745586,510.766679,436.035916,95.682675,47.583890,3.393017,17.287947,16.886588,0.203016,0.137107,4.930851,0.264134,1.118411,0.454697,12.169728,37.400263,35.264298,4.499022,2.024389,6.896315,6.893185,7.687947,6.995868,1.039886,6.966616,0.349770,0.729403
min,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,42.250000,10.250000,51.750000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,14.500000,11.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.030000,17.730000,12.077500,0.402500,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,84.500000,25.000000,890.000000,0.000000,8.500000,2.000000,0.000000,9.500000,10.500000,3.000000,8.000000,2.000000,35.000000,298.500000,217.500000,18.000000,5.500000,1.000000,6.000000,0.000000,0.000000,0.000000,1.000000,0.060000,0.670000,0.145000,4.310000,31.360000,22.180000,2.310000,0.415000,0.050000,0.480000,1.265000,1.065000,0.380000,0.800000,0.165000,0.000000
75%,126.750000,31.000000,2016.000000,1.750000,19.750000,6.000000,2.000000,28.000000,34.750000,14.000000,24.750000,6.000000,89.000000,750.750000,602.000000,83.750000,18.750000,2.750000,15.750000,0.000000,0.110000,0.110000,3.000000,0.247500,1.597500,0.427500,5.680000,45.650000,36.997500,4.970000,1.700000,0.200000,1.207500,2.007500,2.297500,0.970000,1.637500,0.410000,0.000000
max,169.000000,35.000000,3060.000000,12.000000,93.000000,40.000000,16.000000,86.000000,210.000000,57.000000,108.000000,27.000000,204.000000,1984.000000,1666.000000,644.000000,258.000000,21.000000,82.000000,127.000000,1.330000,1.000000,24.000000,1.460000,6.160000,2.470000,90.000000,360.000000,360.000000,25.000000,11.250000,90.000000,90.000000,90.000000,90.000000,11.250000,90.000000,1.760000,4.570000


In [35]:
#We can see that t he 25% cutoff is 51.75 minutes, however, the decimals are not captured within the datasets so we will just round up to 52 as our cutoff point
finalData25 = finalData25.drop(finalData25[finalData25["Mins"] <= 52].index)

In [36]:
finalData25.describe()

,ID,App,Mins,Assists,Shots,shotsOnTarget,Goals,Tackles,Clearances,Interceptions,aerialsWon,Blocks,duelsWon,Passes,accuratePasses,longBalls,Crosses,bigChancesCreated,keyPasses,Saves,goalsPer90,assistsPer90,g+a,g+aPer90,shotsPer90,shotsOnTargetPer90,duelsWonPer90,passesPer90,accuratePassesPer90,longBallsPer90,crossesPer90,bigChancesCreatedPer90,keyPassesPer90,tacklesPer90,clearancesPer90,interceptionsPer90,aerialsWonPer90,blocksPer90,savesPer90
count,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000,127.000000
mean,81.212598,25.677165,1457.771654,1.614173,19.897638,6.921260,2.377953,23.062992,34.062992,11.952756,21.267717,5.165354,68.724409,622.858268,508.456693,79.897638,30.692913,2.732283,15.685039,4.582677,0.139843,0.098031,3.992126,0.238189,1.322756,0.428425,4.688898,37.140472,29.792756,4.772598,1.654409,0.171181,0.940945,1.557953,2.066850,0.716693,1.491969,0.360394,0.231575
std,50.058161,7.939135,911.494989,2.449643,21.176070,9.088285,3.726629,19.457625,38.609101,12.279715,21.083902,5.702636,47.617137,501.654591,435.344231,103.231504,52.887799,3.682754,18.398456,19.419529,0.224266,0.150876,5.342991,0.281268,1.084508,0.480116,2.036330,14.628038,13.933879,4.589013,2.005258,0.239413,0.747363,1.000529,1.689314,0.577249,1.419716,0.337724,0.836614
min,0.000000,4.000000,54.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,14.000000,7.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.090000,10.740000,6.920000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,34.000000,22.000000,565.000000,0.000000,4.000000,1.000000,0.000000,7.500000,8.000000,2.000000,5.000000,1.000000,29.000000,205.000000,156.500000,12.500000,3.000000,0.000000,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.585000,0.080000,3.695000,26.025000,18.200000,1.605000,0.275000,0.000000,0.335000,0.875000,0.760000,0.215000,0.510000,0.110000,0.000000
50%,80.000000,29.000000,1415.000000,1.000000,14.000000,4.000000,1.000000,17.000000,17.000000,8.000000,14.000000,3.000000,65.000000,511.000000,382.000000,37.000000,9.000000,2.000000,10.000000,0.000000,0.060000,0.050000,2.000000,0.150000,0.980000,0.280000,4.700000,34.750000,26.860000,3.350000,0.820000,0.110000,0.850000,1.480000,1.630000,0.640000,1.140000,0.290000,0.000000
75%,128.500000,31.500000,2253.000000,2.000000,26.000000,8.000000,3.000000,36.500000,47.500000,16.000000,30.500000,8.000000,108.000000,824.000000,701.000000,111.000000,32.000000,3.500000,20.000000,0.000000,0.165000,0.145000,5.000000,0.340000,1.945000,0.570000,5.800000,48.470000,40.865000,6.355000,2.275000,0.250000,1.350000,2.110000,2.765000,1.070000,1.800000,0.540000,0.000000
max,167.000000,35.000000,3060.000000,12.000000,93.000000,40.000000,16.000000,86.000000,210.000000,57.000000,108.000000,27.000000,204.000000,1984.000000,1666.000000,644.000000,258.000000,21.000000,82.000000,127.000000,1.330000,1.000000,24.000000,1.460000,6.160000,2.470000,11.250000,88.040000,77.280000,25.000000,9.500000,2.000000,4.000000,5.010000,7.860000,2.580000,9.640000,1.750000,4.570000


In [37]:
#Now that the changes have successfully gone through, we'll save the final database
finalData25.to_csv("data/finalSCRAPEDdata/masterDatabases/2BULIcomplete25.csv")

### 2.3 Quick Observations

__(This part covers every datacleaning file)__

An issue that has arisen with these splits is that the player ID number is now inconsistent within the altered datasets, as removed entries do not pass on their numbers to continue the natural flow of counting up the ID number, I cannot see how this'll affect anything though as the ID is just to distinguish players within the dataset and has no use within the Tableau Datadashboard, so as much of an issue as it is, it is more of a housekeeping nitpick rather than a system breaking issue.

From creating the split datasets, through manual inspection these are the size differences between each dataset.

- 2BULIcomplete = 170 rows
- 2BULIcomplete25 = 127 rows
- 2BULIcomplete90 = 124 rows

We can note that the changes between the datasets are as follows:

- 2BULIcomplete90 has 27.06% less rows than 2BULIcomplete
- 2BULIcomplete25 and 2BULIcomplete90 have a 1.77% ~ differential between them, quite a minor change.

In the case of this league, we can see that the 90 minute and 25% minute cutoff points are almost the same entry wise, leading to a possible minimal difference in later ablation studies.

#### Bundesliga Observations

From creating the split datasets, through manual inspection these are the size differences between each dataset.

- BULIcomplete = 296 rows
- BULIcomplete25 = 222 rows
- BULIcomplete90 = 251 rows

We can see greater differences in not only overall data captured but also in potentially higher quality data, as more players played more minutes. The difference between the 90 dataset is bigger than in the 2BULI datasets too.

Alongside that, directly comparing to this league's domestic counterpart in the lower tiered 2. Bundesliga, we can see a massive increase in players/recorded data.

- BULIcomplete90 has 15.2% less rows than BULIcomplete
- BULIcomplete25 and BULIcomplete90 have a 9.8% ~ differential between them

#### Championship Observations

From creating the split datasets, through manual inspection these are the size differences between each dataset.

- CHAMPcomplete = 327 rows
- CHAMPcomplete25 = 245 rows
- CHAMPcomplete90 = 287 rows

Similarly to the Bundesliga, we have overall more players in every version of the dataset, with another big difference in the 90 dataset compared to the alternatives.

- CHAMPcomplete90 has 12.3% less rows than CHAMPcomplete
- CHAMPcomplete25 and CHAMPcomplete90 have a 12.7% ~ differential between them

#### Eredivisie Observations

From creating the split datasets, through manual inspection these are the size differences between each dataset.

- EREcomplete = 327 rows
- EREcomplete25 = 245 rows
- EREcomplete90 = 248 rows

Similarly to the 2BULI datasets, the difference between the 90 and 25% datasets are miniscule.

- EREcomplete90 has 24.16% less rows than EREcomplete
- EREcomplete25 and EREcomplete90 have a 0.84% ~ differential between them

#### LaLiga Observations

From creating the split datasets, through manual inspection these are the size differences between each dataset.

- LALIGAcomplete = 358 rows
- LALIGAcomplete25 = 268 rows
- LALIGAcomplete90 = 312 rows

LaLiga isn't our first top 5 European League that we have covered so far in our analysis (it was the German Bundesliga), however, this is the first top 5 European League with 20 teams participating, this is reflected in the row count.

- LALIGAcomplete90 has 12.85% less rows than LALIGAcomplete
- LALIGAcomplete25 and LALIGAcomplete90 have a 12.15% ~ differential between them

#### LaLiga2

- 237 rows for main

#### Liga Portugal Observations

From creating the split datasets, through manual inspection these are the size differences between each dataset.

- LIGAPcomplete = 226 rows
- LIGAPcomplete25 = 169 rows
- LIGAPcomplete90 = 178 rows
- LIGAPcomplete90 has 21.24% less rows than LIGAPcomplete
- LIGAPcomplete25 and LIGAPcomplete90 have a 3.76% ~ differential between them

#### Liga Portugal 2 Observations

From creating the split datasets, through manual inspection these are the size differences between each dataset.

- LIGAP2complete = 134 rows
- LIGAP2complete25 = 98 rows
- LIGAP2complete90 = 81 rows

As the lower tier domestic league compared to Liga Portugal 1, much less data has been captured.

Alongside that, as seen below, Liga Portugal 2 is the first league to have less players who played more than 90 minutes in a season, than the 25% cutoff.

- LIGAP2complete90 has 39.55% less rows than LIGAP2complete
- LIGAP2complete25 and LIGAP2complete90 have a -14.55% ~ differential between them

#### Ligue 1 Observations

From creating the split datasets, through manual inspection these are the size differences between each dataset.

- L1complete = 266 rows
- L1complete25 = 199 rows
- L1complete90 = 216 rows

Another top 5 league with an 18 team bracket, it is slightly smaller when compared to its only other counterpart in the top 5, the German Bundesliga.

- L1complete90 has 18.8% less rows than L1complete
- L1complete25 and L1complete90 have a 6.2% ~ differential between them

#### Ligue 2 Observations

From creating the split datasets, through manual inspection these are the size differences between each dataset.

- L2complete = 142 rows
- L2complete25 = 106 rows
- L2complete90 = 120 rows

A steep difference in recorded data to its higher tier domestic league in Ligue 1, this has also occured in the Portuguese leagues.

- L2complete90 has 15.49% less rows than L1complete
- L2complete25 and L2complete90 have a 9.51% ~ differential between them

#### Premier League Observations

From creating the split datasets, through manual inspection these are the size differences between each dataset.

- PLcomplete = 341 rows
- PLcomplete25 = 255 rows
- PLcomplete90 = 308 rows

The Premier League is almost universally considered the biggest football league on the planet, definitely the richest in spending, although not entirely relevant. The observation to point out here is that for the biggest league in the world, it has not used the most players out of the top 5 leagues, as a 20 team championship, so far we can compare it to the Spanish LaLiga, and further down to the Italian  Serie A.

- PLcomplete90 has 9.68% less rows than PLcomplete
- PLcomplete25 and PLcomplete90 have a 15.32% ~ differential between them

#### Premiership Observations

From creating the split datasets, through manual inspection these are the size differences between each dataset.

- PScomplete = 163 rows
- PScomplete25 = 122 rows
- PScomplete90 = 142 rows
- PScomplete90 has 12.88% less rows than PScomplete
- PScomplete25 and PScomplete90 have a 12.12% ~ differential between them

#### Serie A Observations

From creating the split datasets, through manual inspection these are the size differences between each dataset.

- SEAcomplete = 352 rows
- SEAcomplete25 = 264 rows
- SEAcomplete90 = 293 rows

As the third and final top 5 European League sporting a 20 team championship, falling inbetween the English Premier League and the Spanish LaLiga in players recorded over a season.

- SEAcomplete90 has 16.76% less rows than SEAcomplete
- SEAcomplete25 and SEAcomplete90 have a 8.24% differential between them

#### Serie B Observations

From creating the split datasets, through manual inspection these are the size differences between each dataset.

- SEBcomplete = 135
- SEBcomplete25 = 101
- SEBcomplete90 = 121

Another domestic league that is a tier lower in this study, Italy's Serie B follows the now established pattern of lower leagues having less players recorded when compared against the higher tiered domestic league.

- SEBcomplete90 has 10.37% less rows than SEBcomplete
- SEBcomplete25 and SEBcomplete90 have a 14.63% differential between them

### 2.4 Master Database Creation

As we now have the 3 master databases per league, we can continue on to create the final master databases.

As some leagues contain expected stats, and some do not, I will split the databases further, into:

- masterDB containing all leagues with expected stats
- noXMasterDB containing all leagues with no expected stats

Then the same for the 90minute and 25% datasets.

In [38]:
#Calling every dataframe that has been created in separate .ipynb files at this point, following the naming schemes of the saved .csv files
#If a tier isn't mentioned, it is the top tier domestic league of that country

#complete datasets
#for some reason naming 2BULI with a 2 at the beginning is causing problems
twoBULIcomplete = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/2BULIcomplete.csv")        #2. Bundesliga, Germany (second tier) - done in this file
BULIcomplete = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/BULIcomplete.csv")            #Bundesliga, Germany (top tier) - done in "dataCleaningBULI.ipynb"
CHAMPcomplete = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/CHAMPcomplete.csv")          #Championship, England (second tier) - done in "dataCleaningCHAMPIONSHIP.ipynb"
EREcomplete = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/EREcomplete.csv")              #Eredivisie, Netherlands - done in "dataCleaningERE.ipynb"
LALIGAcomplete = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/LALIGAcomplete.csv")        #LaLiga, Spain (top tier) - done in "dataCleaningLALIGA.ipynb"
LALIGA2complete = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/LALIGA2complete.csv")      #LaLiga 2, Spain (second tier) - done in "dataCleaningLALIGA2.ipynb"
LIGAPcomplete = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/LIGAPcomplete.csv")          #Liga Portugal, Portugal (top tier) - done in "dataCleaningLIGAP.ipynb"
LIGAP2complete = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/LIGAP2complete.csv")        #Liga Portugal 2, Portugal (second tier) - done in "dataCleaningLIGAP2.ipynb"
L1complete = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/L1complete.csv")                #Ligue 1, France (top tier) - done in "dataCleaningL1.ipynb"
L2complete = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/L2complete.csv")                #Ligue 2, France (second tier) - done in "dataCleaningL2.ipynb"
PLcomplete = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/PLcomplete.csv")                #Premier League, England (top tier) - done in "dataCleaningPL.ipynb"
PScomplete = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/PScomplete.csv")                #Premiership, Scotland - done in "dataCleaningPS.ipynb"
SEAcomplete = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/SEAcomplete.csv")              #Serie A, Italy (top tier) - done in "dataCleaningSEA.ipynb"
SEBcomplete = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/SEBcomplete.csv")              #Serie b, Italy (second tier) - done in "dataCleaningSEB.ipynb"

#25% datasets
twoBULIcomplete25 = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/2BULIcomplete25.csv")    #No expected stats
BULIcomplete25 = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/BULIcomplete25.csv")
CHAMPcomplete25 = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/CHAMPcomplete25.csv")
EREcomplete25 = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/EREcomplete25.csv")
LALIGAcomplete25 = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/LALIGAcomplete25.csv")
LALIGA2complete25 = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/LALIGA2complete25.csv")  #No expected stats
LIGAPcomplete25 = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/LIGAPcomplete25.csv")
LIGAP2complete25 = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/LIGAP2complete25.csv")    #No Expected Stats
L1complete25 = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/L1complete25.csv")
L2complete25 = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/L2complete25.csv")            #No expected stats
PLcomplete25 = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/PLcomplete25.csv")
PScomplete25 = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/PScomplete25.csv")            #No expected stats
SEAcomplete25 = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/SEAcomplete25.csv")
SEBcomplete25 = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/SEBcomplete25.csv")          #No expected stats

#90minute datasets
twoBULIcomplete90 = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/2BULIcomplete90.csv")
BULIcomplete90 = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/BULIcomplete90.csv")
CHAMPcomplete90 = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/CHAMPcomplete90.csv")
EREcomplete90 = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/EREcomplete90.csv")
LALIGAcomplete90 = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/LALIGAcomplete90.csv")
LALIGA2complete90 = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/LALIGA2complete90.csv")
LIGAPcomplete90 = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/LIGAPcomplete90.csv")
LIGAP2complete90 = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/LIGAP2complete90.csv")
L1complete90 = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/L1complete90.csv")
L2complete90 = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/L2complete90.csv")
PLcomplete90 = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/PLcomplete90.csv")
PScomplete90 = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/PScomplete90.csv")
SEAcomplete90 = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/SEAcomplete90.csv")
SEBcomplete90 = pd.read_csv("data/finalSCRAPEDdata/masterDatabases/SEBcomplete90.csv")

As we have some datasets that contain no expected statistics (noted above), we will further the master database splits into the "complete" datasets containing all leagues with expected statistics, and "noX" datasets containing all leagues that were left out from the complete versions.

My reasoning for this is because the addition of 4 datasets containing 0s across all their rows would massively skew the data later on in analysis and machine learning training, hence making the study pointless, the split serves as a further look into how important expected stats are versus if their absence is noticeable.

The issue of player ID pops up here again, as it won't be counting up consistently because of the omission of specific datasets, however that is only a problem in the bookkeeping rather than something the user will see issues with.

In [39]:
#merging everything - THIS CODE IS HERE JUST AS REFERENCE TO MERGE THE ABOVE LATER
#first we do the normal dataset
completeFull = [BULIcomplete, CHAMPcomplete, EREcomplete, LALIGAcomplete, LIGAPcomplete, L1complete, PLcomplete, SEAcomplete]
#25% dataset
complete25 = [BULIcomplete25, CHAMPcomplete25, EREcomplete25, LALIGAcomplete25, LIGAPcomplete25, L1complete25, PLcomplete25, SEAcomplete25]
#<90 min dataset
complete90 = [BULIcomplete90, CHAMPcomplete90, EREcomplete90, LALIGAcomplete90, LIGAPcomplete90, L1complete90, PLcomplete90, SEAcomplete90]

#dataframe creations
#main
masterDB = pd.concat(completeFull, ignore_index=True)
#25%
masterDB25 = pd.concat(complete25, ignore_index=True)
#<90min
masterDB90 = pd.concat(complete90, ignore_index=True)

#now for the nonexpected stats
noXFull = [twoBULIcomplete, LALIGA2complete, LIGAP2complete, L2complete, PScomplete, SEBcomplete]
#25% dataset
noX25 = [twoBULIcomplete25, LALIGA2complete25, LIGAP2complete25,  L2complete25, PScomplete25, SEBcomplete25]
#<90 min dataset
noX90 = [twoBULIcomplete90, LALIGA2complete90, LIGAP2complete90,  L2complete90, PScomplete90, SEBcomplete90]

#nonexepcted stat dataframes - main first
noXMasterDB = pd.concat(noXFull, ignore_index=True)
#25%
noXMasterDB25 = pd.concat(noX25, ignore_index=True)
#<90
noXMasterDB90 = pd.concat(noX90, ignore_index=True)

#quick checks
#masterDB.head(-1) #we can see 2 separate leagues, confirming the correct merge
noXMasterDB90.head(-1) #same as above

,Unnamed: 0,ID,Position,League,Name,Team,App,Mins,Assists,Shots,shotsOnTarget,Goals,Tackles,Clearances,Interceptions,aerialsWon,Blocks,duelsWon,Passes,accuratePasses,longBalls,Crosses,bigChancesCreated,keyPasses,Saves,goalsPer90,assistsPer90,g+a,g+aPer90,shotsPer90,shotsOnTargetPer90,duelsWonPer90,passesPer90,accuratePassesPer90,longBallsPer90,crossesPer90,bigChancesCreatedPer90,keyPassesPer90,tacklesPer90,clearancesPer90,interceptionsPer90,aerialsWonPer90,blocksPer90,savesPer90
0,0,0,Defender,2. Bundesliga,E. Valentini,1. FC Nürnberg,14,90,1,1.0,0,0,3,2,0,1,0,9,66,56,12,8,2,4,0.0,0.00,1.00,1,1.00,1.00,0.00,9.00,66.00,56.00,12.00,8.00,2.00,4.00,3.00,2.00,0.00,1.00,0.00,0.0
1,1,1,Defender,2. Bundesliga,S. Jung,Karlsruher SC,31,2712,1,26.0,6,2,38,64,22,13,10,64,1047,813,135,106,3,21,0.0,0.07,0.03,3,0.10,0.86,0.20,2.12,34.75,26.98,4.48,3.52,0.10,0.70,1.26,2.12,0.73,0.43,0.33,0.0
2,2,2,Defender,2. Bundesliga,M. Zimmermann,Fortuna Düsseldorf,33,2221,2,24.0,10,3,64,62,24,33,6,126,1212,1023,69,58,3,18,0.0,0.12,0.08,5,0.20,0.97,0.41,5.11,49.11,41.45,2.80,2.35,0.12,0.73,2.59,2.51,0.97,1.34,0.24,0.0
3,3,3,Defender,2. Bundesliga,T. Kalas,FC Schalke 04,27,1319,1,6.0,2,1,16,95,27,62,0,95,699,580,86,9,1,6,0.0,0.07,0.07,2,0.14,0.41,0.14,6.48,47.70,39.58,5.87,0.61,0.07,0.41,1.09,6.48,1.84,4.23,0.00,0.0
4,4,4,Defender,2. Bundesliga,M. Halstenberg,Hannover 96,28,2492,2,26.0,8,3,42,99,37,42,8,110,1536,1303,222,37,2,15,0.0,0.11,0.07,5,0.18,0.94,0.29,3.97,55.47,47.06,8.02,1.34,0.07,0.54,1.52,3.58,1.34,1.52,0.29,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
782,128,3467,Midfielder,Serie B,R. Floriani Mussolini,Cremonese,37,2890,2,16.0,8,1,44,92,21,43,1,119,529,351,68,92,5,18,0.0,0.03,0.06,3,0.09,0.50,0.25,3.71,16.47,10.93,2.12,2.87,0.16,0.56,1.37,2.87,0.65,1.34,0.03,0.0
783,129,3468,Midfielder,Serie B,J. Oyono,Frosinone,38,1585,0,6.0,3,1,36,49,6,21,0,105,596,484,54,43,1,18,0.0,0.06,0.00,1,0.06,0.34,0.17,5.96,33.84,27.48,3.07,2.44,0.06,1.02,2.04,2.78,0.34,1.19,0.00,0.0
784,130,3469,Midfielder,Serie B,S. Angori,Pisa,38,2354,3,28.0,5,2,27,72,9,40,11,133,759,580,107,192,9,55,0.0,0.08,0.11,5,0.19,1.07,0.19,5.08,29.02,22.18,4.09,7.34,0.34,2.10,1.03,2.75,0.34,1.53,0.42,0.0
785,131,3470,Midfielder,Serie B,I. Vural,Pisa,36,1196,1,14.0,2,0,25,20,8,15,2,91,414,361,25,2,2,10,0.0,0.00,0.08,1,0.08,1.05,0.15,6.85,31.15,27.17,1.88,0.15,0.15,0.75,1.88,1.51,0.60,1.13,0.15,0.0


We can see the column "Unnamed: 0" has appeared, we'll quickly drop that from all datasets

In [40]:
#what to drop
unnamedDrop = ["Unnamed: 0"]

#every dataframe
for df in [masterDB, masterDB25, masterDB90, noXMasterDB, noXMasterDB25, noXMasterDB90]:
    df.drop(columns=unnamedDrop, inplace=True)

#quick check
#masterDB.head(-1)
noXMasterDB90.head(-1) #works

,ID,Position,League,Name,Team,App,Mins,Assists,Shots,shotsOnTarget,Goals,Tackles,Clearances,Interceptions,aerialsWon,Blocks,duelsWon,Passes,accuratePasses,longBalls,Crosses,bigChancesCreated,keyPasses,Saves,goalsPer90,assistsPer90,g+a,g+aPer90,shotsPer90,shotsOnTargetPer90,duelsWonPer90,passesPer90,accuratePassesPer90,longBallsPer90,crossesPer90,bigChancesCreatedPer90,keyPassesPer90,tacklesPer90,clearancesPer90,interceptionsPer90,aerialsWonPer90,blocksPer90,savesPer90
0,0,Defender,2. Bundesliga,E. Valentini,1. FC Nürnberg,14,90,1,1.0,0,0,3,2,0,1,0,9,66,56,12,8,2,4,0.0,0.00,1.00,1,1.00,1.00,0.00,9.00,66.00,56.00,12.00,8.00,2.00,4.00,3.00,2.00,0.00,1.00,0.00,0.0
1,1,Defender,2. Bundesliga,S. Jung,Karlsruher SC,31,2712,1,26.0,6,2,38,64,22,13,10,64,1047,813,135,106,3,21,0.0,0.07,0.03,3,0.10,0.86,0.20,2.12,34.75,26.98,4.48,3.52,0.10,0.70,1.26,2.12,0.73,0.43,0.33,0.0
2,2,Defender,2. Bundesliga,M. Zimmermann,Fortuna Düsseldorf,33,2221,2,24.0,10,3,64,62,24,33,6,126,1212,1023,69,58,3,18,0.0,0.12,0.08,5,0.20,0.97,0.41,5.11,49.11,41.45,2.80,2.35,0.12,0.73,2.59,2.51,0.97,1.34,0.24,0.0
3,3,Defender,2. Bundesliga,T. Kalas,FC Schalke 04,27,1319,1,6.0,2,1,16,95,27,62,0,95,699,580,86,9,1,6,0.0,0.07,0.07,2,0.14,0.41,0.14,6.48,47.70,39.58,5.87,0.61,0.07,0.41,1.09,6.48,1.84,4.23,0.00,0.0
4,4,Defender,2. Bundesliga,M. Halstenberg,Hannover 96,28,2492,2,26.0,8,3,42,99,37,42,8,110,1536,1303,222,37,2,15,0.0,0.11,0.07,5,0.18,0.94,0.29,3.97,55.47,47.06,8.02,1.34,0.07,0.54,1.52,3.58,1.34,1.52,0.29,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
782,3467,Midfielder,Serie B,R. Floriani Mussolini,Cremonese,37,2890,2,16.0,8,1,44,92,21,43,1,119,529,351,68,92,5,18,0.0,0.03,0.06,3,0.09,0.50,0.25,3.71,16.47,10.93,2.12,2.87,0.16,0.56,1.37,2.87,0.65,1.34,0.03,0.0
783,3468,Midfielder,Serie B,J. Oyono,Frosinone,38,1585,0,6.0,3,1,36,49,6,21,0,105,596,484,54,43,1,18,0.0,0.06,0.00,1,0.06,0.34,0.17,5.96,33.84,27.48,3.07,2.44,0.06,1.02,2.04,2.78,0.34,1.19,0.00,0.0
784,3469,Midfielder,Serie B,S. Angori,Pisa,38,2354,3,28.0,5,2,27,72,9,40,11,133,759,580,107,192,9,55,0.0,0.08,0.11,5,0.19,1.07,0.19,5.08,29.02,22.18,4.09,7.34,0.34,2.10,1.03,2.75,0.34,1.53,0.42,0.0
785,3470,Midfielder,Serie B,I. Vural,Pisa,36,1196,1,14.0,2,0,25,20,8,15,2,91,414,361,25,2,2,10,0.0,0.00,0.08,1,0.08,1.05,0.15,6.85,31.15,27.17,1.88,0.15,0.15,0.75,1.88,1.51,0.60,1.13,0.15,0.0


In [41]:
#fill NaN rows with 0 (reused code)
for df in [masterDB, masterDB25, masterDB90, noXMasterDB, noXMasterDB25, noXMasterDB90]:
    numberColumns = finalData.select_dtypes(include=["float64", "int64"]).columns
    df[numberColumns] = df[numberColumns].fillna(0)
    #round decimals to 2
    df = df.round(decimals=2)

In [42]:
masterDB.isnull().sum()

ID                        0
Position                  0
League                    0
Name                      0
Team                      0
App                       0
Mins                      0
Assists                   0
Shots                     0
shotsOnTarget             0
Goals                     0
xG                        0
npxG                      0
xA                        0
Tackles                   0
Clearances                0
Interceptions             0
aerialsWon                0
Blocks                    0
duelsWon                  0
Passes                    0
accuratePasses            0
longBalls                 0
Crosses                   0
bigChancesCreated         0
keyPasses                 0
Saves                     0
goalsPer90                0
assistsPer90              0
g+a                       0
g+aPer90                  0
shotsPer90                0
shotsOnTargetPer90        0
duelsWonPer90             0
passesPer90               0
accuratePassesPer90 

We can now go ahead and save these datasets as csv files.

In [43]:
#normal datasets
masterDB.to_csv("data/finalSCRAPEDdata/masterDatabases/masterDB.csv")
masterDB25.to_csv("data/finalSCRAPEDdata/masterDatabases/masterDB25.csv")
masterDB90.to_csv("data/finalSCRAPEDdata/masterDatabases/masterDB90.csv")

#non expected datasets
noXMasterDB.to_csv("data/finalSCRAPEDdata/masterDatabases/noXMasterDB.csv")
noXMasterDB25.to_csv("data/finalSCRAPEDdata/masterDatabases/noXMasterDB25.csv")
noXMasterDB90.to_csv("data/finalSCRAPEDdata/masterDatabases/noXMasterDB90.csv")

With further progression into the project, masterDB25, masterDB90, noXMasterDB25 and noXMasterDB90 are not used pas this point.

Now that we have created our master databases for all leagues, we can move onto machineLearning.ipynb to train our algorithms and do in-depth data analysis before applying that knowledge to the final user product of a Tableau Datadashboard.